# Dataset Creation
This notebook:
- load correlation matrices from a `.pt` file
- select only matrices in the range `[MATRIX_START, MATRIX_END)`
- split matrices into train, validation, and test sets
- save the three datasets under `data/processed/dataset`

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import torch
import math

np.set_printoptions(suppress=True, precision=4)

## Step 1: Load Data and Select Matrix Range
Load correlation matrices from a `.pt` file and keep only matrices in `[MATRIX_START, MATRIX_END)` (same behavior as `05_linearAE.ipynb`).

In [2]:
FILE_NAME = 'data_00_20'
WINDOW_SIZE = 724
STRIDE = 1
MY_FILE_NAME = f'{FILE_NAME}_w{WINDOW_SIZE}_s{STRIDE}.pt'

if 'google.colab' in sys.modules:
    print('Environment detected: Google Colab')
    IS_COLAB = True
else:
    print('Environment detected: Local (PC)')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    processed_root = Path('/content/drive/MyDrive/dataset_tesi')
else:
    project_root = Path.cwd().resolve().parent
    processed_root = project_root / 'data' / 'processed' / FILE_NAME

corr_matrices_dir = processed_root / 'correlation_matrices'
if not corr_matrices_dir.exists():
    corr_matrices_dir = processed_root

selected_file = corr_matrices_dir / MY_FILE_NAME
if not selected_file.exists():
    raise FileNotFoundError(f"File '{MY_FILE_NAME}' not found in: {corr_matrices_dir.resolve()}")

# Select the matrix range to use: [MATRIX_START, MATRIX_END)
MATRIX_START = 486  # number or None
MATRIX_END = None  # number or None; e.g., 500 to use only first 500 matrices

print(f'Selected file: {selected_file.name}')
print(f'Matrix range requested: [{MATRIX_START}, {MATRIX_END})')

Environment detected: Local (PC)
Selected file: data_00_20_w724_s1.pt
Matrix range requested: [486, None)


In [3]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), meta

# --- CARICAMENTO ---
corr_tensor, meta = load_corr_payload(selected_file)

orig_n = corr_tensor.shape[0]
start_idx = 0 if MATRIX_START is None else int(MATRIX_START)
end_idx = orig_n if MATRIX_END is None else int(MATRIX_END)

if not (0 <= start_idx < end_idx <= orig_n):
    raise ValueError(f'Invalid matrix range [{start_idx}, {end_idx}) for dataset size {orig_n}')

# --- TAGLIO DEI DATI ---
corr_tensor = corr_tensor[start_idx:end_idx]

if 'window_ranges' in meta and len(meta['window_ranges']) == orig_n:
    meta['window_ranges'] = meta['window_ranges'][start_idx:end_idx]

# ==========================================
# NUOVO: CALCOLO CHOLESKY DECOMPOSITION
# ==========================================
CHOLESKY_JITTER = 1e-6
n_assets = corr_tensor.shape[1]

print("\nCalcolo scomposizione di Cholesky...")
# Creazione matrice identità per stabilizzare la scomposizione
jitter_matrix = torch.eye(n_assets, dtype=corr_tensor.dtype, device=corr_tensor.device) * CHOLESKY_JITTER
jittered_corr = corr_tensor + jitter_matrix

try:
    # torch.linalg.cholesky restituisce direttamente il fattore triangolare inferiore
    chol_tensor = torch.linalg.cholesky(jittered_corr)
    print("✓ Scomposizione di Cholesky completata con successo.")
except RuntimeError as e:
    raise RuntimeError(f"Errore algebrico durante Cholesky. Potrebbe servire un JITTER maggiore. Dettagli: {e}")
# ==========================================
# ==========================================
# ESTRAZIONE METADATI (Timestamps, ecc.)
# ==========================================
# Estraiamo in variabili "sciolte" così lo script di partizionamento le trova
timestamps = meta.get('timestamps', [])
window_ranges = meta.get('window_ranges', [])
window_length = meta.get('window_length', 252)
stride = meta.get('stride', 5)


# --- OUTPUT DI CONTROLLO ---
print(f"{'='*50}")
print("DATASET LOADED & SLICED")
print(f"{'='*50}")
print(f'corr_tensor shape : {tuple(corr_tensor.shape)}')
print(f'chol_tensor shape : {tuple(chol_tensor.shape)}')
print(f'using matrices    : [{start_idx}, {end_idx}) out of {orig_n}')
print(f'dtype             : {corr_tensor.dtype}')

# Se abbiamo i timestamp, stampiamo la prima e l'ultima data per controllo
if timestamps and window_ranges:
    first_matrix_start = timestamps[window_ranges[0][0]]
    first_matrix_end   = timestamps[window_ranges[0][1] - 1]
    last_matrix_start  = timestamps[window_ranges[-1][0]]
    last_matrix_end    = timestamps[window_ranges[-1][1] - 1]
    
    print("-" * 50)
    print(f"Total timestamps  : {len(timestamps)}")
    print(f"First matrix date : {first_matrix_start} to {first_matrix_end}")
    print(f"Last matrix date  : {last_matrix_start} to {last_matrix_end}")
else:
    print("-" * 50)
    print("Warning: 'timestamps' or 'window_ranges' missing from .pt file metadata!")
print(f"{'='*50}\n")


Calcolo scomposizione di Cholesky...
✓ Scomposizione di Cholesky completata con successo.
DATASET LOADED & SLICED
corr_tensor shape : (4124, 362, 362)
chol_tensor shape : (4124, 362, 362)
using matrices    : [486, 4610) out of 4610
dtype             : torch.float32
--------------------------------------------------
Total timestamps  : 5333
First matrix date : 2001-10-11 to 2004-08-26
Last matrix date  : 2018-02-28 to 2021-01-12



## Step 2: Create Train/Validation/Test Splits
Random sampling of matrices (no temporal gaps) to build train, validation, and test sets.

In [4]:
# --- PARAMETRI ---
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.2
TEST_FRACTION = 0.1

RANDOM_SEED = 42

# Verifica delle frazioni
if abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) > 1e-6:
    raise ValueError("Le frazioni di Train, Val e Test devono sommare a 1.0")

# Setup iniziale
corr_np = corr_tensor.numpy().astype(np.float32)
chol_np = chol_tensor.numpy().astype(np.float32) # <-- Aggiunto
n_matrices, n_assets, _ = corr_np.shape

# ==========================================
# PARTIZIONAMENTO CASUALE SENZA GAP TEMPORALI
# ==========================================
n_train = int(n_matrices * TRAIN_FRACTION)
n_val = int(n_matrices * VAL_FRACTION)
n_test = n_matrices - n_train - n_val

if min(n_train, n_val, n_test) <= 0:
    raise ValueError(f"Dataset troppo piccolo ({n_matrices}) per il random split con le frazioni date.")

rng = np.random.default_rng(RANDOM_SEED)
perm_idx = rng.permutation(n_matrices)

train_idx = perm_idx[:n_train]
val_idx = perm_idx[n_train:n_train + n_val]
test_idx = perm_idx[n_train + n_val:]

# Estrazione tensori Correlazione
train_corr = torch.from_numpy(corr_np[train_idx])
val_corr = torch.from_numpy(corr_np[val_idx])
test_corr = torch.from_numpy(corr_np[test_idx])

# Estrazione tensori Cholesky
train_chol = torch.from_numpy(chol_np[train_idx])
val_chol = torch.from_numpy(chol_np[val_idx])
test_chol = torch.from_numpy(chol_np[test_idx])

# ==========================================
# ESTRAZIONE DATE PER I PRINT (min/max indice)
# ==========================================
def split_stats(idxs: np.ndarray):
    idx_min = int(np.min(idxs))
    idx_max = int(np.max(idxs))
    if timestamps and window_ranges:
        start_date = timestamps[window_ranges[idx_min][0]]
        end_date = timestamps[window_ranges[idx_max][1] - 1]
    else:
        start_date = None
        end_date = None
    return idx_min, idx_max, start_date, end_date

train_min, train_max, train_start_date, train_end_date = split_stats(train_idx)
val_min, val_max, val_start_date, val_end_date = split_stats(val_idx)
test_min, test_max, test_start_date, test_end_date = split_stats(test_idx)

# --- OUTPUT STATISTICHE ---
print(f"{'='*90}")
print("DATASET SPLIT (random sampling)")
print(f"{'='*90}")
print(f"Total historical matrices    : {n_matrices}")
print(f"Random seed                  : {RANDOM_SEED}")
print("-" * 90)
print(f"Train shape: {str(tuple(train_corr.shape)):<14} | Chol: {str(tuple(train_chol.shape)):<14}")
print(f"Val shape  : {str(tuple(val_corr.shape)):<14} | Chol: {str(tuple(val_chol.shape)):<14}")
print(f"Test shape : {str(tuple(test_corr.shape)):<14} | Chol: {str(tuple(test_chol.shape)):<14}")
print(f"{'='*90}")

DATASET SPLIT (random sampling)
Total historical matrices    : 4124
Random seed                  : 42
------------------------------------------------------------------------------------------
Train shape: (2886, 362, 362) | Chol: (2886, 362, 362)
Val shape  : (824, 362, 362) | Chol: (824, 362, 362)
Test shape : (414, 362, 362) | Chol: (414, 362, 362)


## Step 3: Save Train/Validation/Test Datasets
Save the `all.pt` file with all matrices in original index order, then save the three split `.pt` files and the summary `.json` inside a dedicated subfolder under `data/processed/{FILE_NAME}/dataset`.

In [5]:
dataset_root_dir = processed_root / 'dataset'
dataset_root_dir.mkdir(parents=True, exist_ok=True)

range_tag = f'range_{start_idx}_{end_idx}'
split_tag = f'train{int(TRAIN_FRACTION * 100)}_val{int(VAL_FRACTION * 100)}_test{int(TEST_FRACTION * 100)}'
base_name = selected_file.stem
dataset_dir = dataset_root_dir / base_name 
dataset_dir.mkdir(parents=True, exist_ok=True)


all_indices = np.arange(start_idx, end_idx)
all_payload = {
    'corr_tensor': corr_tensor.clone(),
    'chol_tensor': chol_tensor.clone(),
    'indices': all_indices.tolist(),
    'split': 'all',
}

split_payloads = {
    'train': {
        'corr_tensor': train_corr.clone(),
        'chol_tensor': train_chol.clone(),
        'indices': train_idx.tolist(),
    },
    'val': {
        'corr_tensor': val_corr.clone(),
        'chol_tensor': val_chol.clone(),
        'indices': val_idx.tolist(),
    },
    'test': {
        'corr_tensor': test_corr.clone(),
        'chol_tensor': test_chol.clone(),
        'indices': test_idx.tolist(),
    },
}

common_meta = {
    'source_file': str(selected_file),
    'matrix_range': {'start_idx': int(start_idx), 'end_idx': int(end_idx)},
    'matrix_shape': [int(n_assets), int(n_assets)],
    'split_fractions': {'train_fraction': float(TRAIN_FRACTION), 'val_fraction': float(VAL_FRACTION), 'test_fraction': float(TEST_FRACTION)},
    'base_name': base_name,
    'range_tag': range_tag,
    'split_tag': split_tag,
    'cholesky_jitter': float(CHOLESKY_JITTER),
    'tickers': meta.get('tickers', None),
}

saved_paths = {}
all_path = dataset_dir / 'all.pt'
torch.save(
    {
        **all_payload,
        'meta': {
            **common_meta,
            'split_size': int(all_payload['corr_tensor'].shape[0]),
        },
    },
    all_path,
)
saved_paths['all'] = all_path

for split_name, payload in split_payloads.items():
    output_path = dataset_dir / f'{split_name}.pt'
    torch.save(
        {
            **payload,
            'split': split_name,
            'meta': {
                **common_meta,
                'split_size': int(payload['corr_tensor'].shape[0]),
            },
        },
        output_path,
    )
    saved_paths[split_name] = output_path

summary_path = dataset_dir / 'dataset_info.json'
summary_payload = {
    'base_name': base_name,
    'dataset_dir': str(dataset_dir),
    'files': {k: str(v) for k, v in saved_paths.items()},
    'sizes': {
        'all': int(corr_tensor.shape[0]),
        'train': int(len(train_idx)),
        'val': int(len(val_idx)),
        'test': int(len(test_idx)),
    },
    'meta': common_meta,
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=4)

print('Saved dataset files inside folder:')
for split_name, output_path in saved_paths.items():
    print(f' - {split_name}: {output_path}')
print(f' - summary: {summary_path}')

Saved dataset files inside folder:
 - all: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w724_s1\all.pt
 - train: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w724_s1\train.pt
 - val: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w724_s1\val.pt
 - test: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w724_s1\test.pt
 - summary: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\data\processed\data_00_20\dataset\data_00_20_w724_s1\dataset_info.json
